<img src="img/pandora2d_logo.png" width="500">

# Pandora2D : a coregistration framework

# Usage of the ambiguity method

<div class="alert alert-block alert-warning">
    This initial version, available in Pandora2d 1.1.0, should not be used with Pandora correlation metrics (SAD, SSD, ZNCC_PYTHON, MC_CNN). An update in a future version will resolve this issue. </br>
</div>

This metric is related to a cost curve property and aims to qualify whether a point is ambiguous or not. From one pixel, the ambiguity is computed by the following formula :

$$
\mathrm{Amb}(x,y,\eta) = \mathrm{Card}\left( \left\{ d \in [d_{\min}, d_{\max}] \mid cv(x,y,d) < \min_{d}(cv(x,y,d)) + \eta \right\} \right)
$$

where $cv(x,y,d)$ is the cost value at pixel $(x,y)$ for disparity $d$ in disparity range $[d_{\min}, d_{\max}]$.

From the previous equation, ambiguity integral measure is derived and it is defined as the area under the ambiguity curve. Then, ambiguity integral measure is converted into a confidence measure :

$$
\mathrm{ConfidenceAmbiguity}(x,y) = 1 - \mathrm{AmbiguityIntegral}(x,y)
$$

The ambiguity integral is by default normalized. However, the user may choose not to perform this normalization by setting the normalization parameter to $false$.

[Sarrazin, E., Cournet, M., Dumas, L., Defonte, V., Fardet, Q., Steux, Y., Jimenez Diaz, N., Dubois, E., Youssefi, D., Buffe, F., 2021. Ambiguity concept in stereo matching pipeline. ISPRS - International Archives of the Photogrammetry, Remote Sensing and Spatial Information Sciences.](https://isprs-archives.copernicus.org/articles/XLIII-B2-2021/383/2021/)


#### Imports and external functions

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from pathlib import Path
from pprint import pprint

from snippets.utils import *

# Pandora2D execution options with state machine

#### Imports of pandora2d

In [ ]:
# Load pandora2d imports
from pandora2d import run_pandora2d
from pandora2d.check_configuration import check_conf
from pandora2d.img_tools import create_datasets_from_inputs
from pandora2d.state_machine import Pandora2DMachine

#### Load and visualize input data 

Provide image path

In [ ]:
# Paths to left and right images
img_left_path = "data/left.tif"
img_right_path = "data/right.tif"

Provide output directory to write results

In [ ]:
output_dir = Path.cwd() / "output"
# If necessary, create output dir
output_dir.mkdir(exist_ok=True, parents=True)

Convert input data to dataset

In [ ]:
input_config = {
    "left": {
        "img": img_left_path,
        "nodata": np.nan,
    },
    "right": {
        "img": img_right_path,
        "nodata": np.nan,
    },
    "col_disparity": {"init": 0, "range": 3},
    "row_disparity": {"init": 0, "range": 3},
}

Create datasets

In [ ]:
image_datasets = create_datasets_from_inputs(input_config=input_config)

Visualize input data

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax0 = fig.add_subplot(1, 2, 1)
ax0.imshow(image_datasets.left["im"].data)
plt.title("Left image");
ax1 = fig.add_subplot(1, 2, 2)
ax1.imshow(image_datasets.right["im"].data)
plt.title("Right image");

#### Instantiate the machine

In [ ]:
pandora2d_machine = Pandora2DMachine()

#### Define pipeline configurations

In [ ]:
user_cfg = {
    "input": input_config,
    "pipeline": {
        "matching_cost": {
            "matching_cost_method": "zncc",
            "window_size": 7,
        },
        "cost_volume_confidence":
        {
            "confidence_method": "ambiguity",
            "eta_max": 0.7,
            "eta_step": 0.01
        },
        "disparity": {
            "disparity_method": "wta",
            "invalid_disparity": -9999,
        },
    },
    "output": {
        "path": "outputs/usage_cost_volume_confidence",
    },
}

<div style="border-left: 4px solid #f39c12; padding: 10px; background-color: #fff3cd;">
<b>Note :</b> The ambiguity integral is normalized by default.
However, the user can choose to disable this normalization by setting the normalization parameter to <code>false</code>.
</div>

#### Check the user configuration

In [ ]:
cfg = check_conf(user_cfg, pandora2d_machine)
pprint(cfg)

#### Run Pandora2D machine

In [ ]:
dataset, completed_cfg = run_pandora2d(pandora2d_machine, cfg)

#### Visualize validity_mask and confidence_measure maps

The map below shows valid points as 0 (valid + pseudo-valid) and invalid points as 1.

In [ ]:
result_validity_mask = (dataset["validity"].data[:, :, 0] + dataset["validity"].data[:, :, 1] == 2).astype(int)
cmap_gr = LinearSegmentedColormap.from_list('green_red', ['green', 'white', 'red'])
plot_image(result_validity_mask, "validity_mask map", output_dir, cmap=cmap_gr)

In [ ]:
fig = plt.figure()
plt.title("Confidence measure map")
plt.imshow(dataset["confidence_measure"].data, cmap=pandora_cmap(), vmin=0, vmax=1)
plt.colorbar();